In [11]:
import pandas as pd
import re
import os

# 1. 정규식 패턴 정의
pattern_count = r"몇|개입|개수|개인"
# 🚨 [추가] 선택지에서 "숫자+단위"를 잡아내는 강력한 정규식 (이상/미만, 권, 캔 등 포함)
pattern_count_choice = r"\d+\s*(개|권|장|병|캔|봉지|통|박스|이상|미만)"
pattern_materials = r"플라스틱|페트|종이|박스|골판지|유리|금속|철|캔|알루미늄|비닐|봉지|팩|스티로폼"
pattern_ask_mat = r"재질은|분류는|무슨\s*재질|어떤\s*재질|재질이\s*무엇|무엇으로\s*만들어|종류는"
pattern_ask_obj = r"무엇|어느|어떤\s*것"

# 🚨 [수정] 질문(text) 하나만 받던 것을, 행(Row) 전체를 받도록 변경
def classify_question_advanced(row):
    q_text = str(row['question']) if pd.notna(row['question']) else "Other"
    
    # a, b, c, d 선택지를 하나의 텍스트로 결합
    choices_text = f"{row.get('a','')} {row.get('b','')} {row.get('c','')} {row.get('d','')}"
    
    # 질문과 선택지 각각에서 카운팅 패턴 확인
    is_count_q = bool(re.search(pattern_count, q_text))
    is_count_c = bool(re.search(pattern_count_choice, choices_text))
    
    # [로직 1] 카운팅 완벽 포획: 질문에 '몇'이 있거나, 선택지가 수량(n개) 형태일 때 (재질 언급 유무 무시하고 YOLO로)
    if is_count_q or is_count_c:
        return "YOLO"
        #return "Count_Mat"
        
    # 재질 및 기타 질문 판별
    has_mat_q = bool(re.search(pattern_materials, q_text))
    asks_mat = bool(re.search(pattern_ask_mat, q_text))
    asks_obj = bool(re.search(pattern_ask_obj, q_text))
    
    # 🚨 [추가] 선택지 내 재질 단어 등장 여부 확인
    mat_in_choices = len(re.findall(pattern_materials, choices_text))

    return "VLM"
    '''
    # [로직 2] 재질 묻기(Ask_Mat) 오분류 방어
    if asks_mat:
        # 질문은 '종류는?' 인데 선택지에 재질 단어가 하나도 없으면 물건(Obj)을 묻는 것임 (예: 딸기잼 상자)
        if mat_in_choices == 0:
            return "Ask_Obj"
        return "Ask_Mat"
        
    # [로직 3] 물건 묻기(Ask_Obj) 오분류 방어
    elif asks_obj and has_mat_q:
        # 질문은 '무엇인가요?' 인데 선택지가 온통 재질 이름(페트병, 캔 등)인 경우
        if mat_in_choices >= 2:
            return "Ask_Mat"
        return "Ask_Obj"
        
    # [로직 4] 그 외 위치, 색깔 등
    return "Other"
    '''

def process_and_save_csv(file_path):
    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return
        
    df = pd.read_csv(file_path).head(100)
    
    # 🚨 [수정] apply 함수에 axis=1 옵션을 주어 row(행) 단위로 함수에 던져줌
    df['class'] = df.apply(classify_question_advanced, axis=1)
    
    new_file_path = file_path.replace('.csv', '_class.csv')
    df.to_csv(new_file_path, index=False, encoding='utf-8-sig')
    print(f"저장 완료: {new_file_path} (총 {len(df)}행)")

if __name__ == "__main__":
    target_files = ['train.csv', 'dev.csv', 'test.csv']
    
    for file in target_files:
        process_and_save_csv(file)
        
    print("모든 파일의 섬세한 분류 및 저장이 완료되었습니다.")

저장 완료: train_class.csv (총 100행)
저장 완료: dev_class.csv (총 100행)
저장 완료: test_class.csv (총 100행)
모든 파일의 섬세한 분류 및 저장이 완료되었습니다.
